## Incorporating Automatic Differentiation in `Tidy3D`

With that basic introduction to automatic differentiation using `autograd`, we can now show how `Tidy3D` lets us do the same thing but where our functions can now involve setting up, running, and postprocessing a `tidy3d.Simulation`.

In [ ]:
import os
import numpy as np
import tidy3d as td
import tidy3d.web as web
import matplotlib.pyplot as plt
import autograd.numpy as anp
import autograd as ag
from tidy3d.web.core.environment import Env, dev

Env.set_current(dev)
web.configure("TIDY3D_DEV_API_KEY")

td.config.logging_level = "INFO"

# Transformed Geometry

In [2]:
def make_simulation(center: tuple, size: tuple, scale: tuple, eps: float) -> td.Simulation:
    """Makes a simulation with a variable scatter width, height, and relative permittivity."""

    wavelength = 1.0
    freq0 = td.C_0 / wavelength
    dl = 0.02

    # Static structure
    waveguide = td.Structure(
        geometry=td.Box(size=(td.inf, 0.3, 0.2)), medium=td.Medium(permittivity=2.0)
    )

    # Forward source
    mode_src = td.ModeSource(
        size=(0, 1.5, 1.5),
        center=(-0.9, 0, 0),
        mode_index=0,
        source_time=td.GaussianPulse(freq0=freq0, fwidth=freq0 / 10),
        direction="+",
    )

    # Monitor
    mode_mnt = td.ModeMonitor(
        size=(0, 1.5, 1.5),
        center=(+0.9, 0, 0),
        mode_spec=mode_src.mode_spec,
        freqs=[freq0],
        name="mode",
    )

    # Geometry and transformation
    base_geometry = td.Box(center=center, size=size)
    transformed_geometry = base_geometry.scaled(x=scale[0], y=scale[1], z=scale[2])

    # Scatterer structure
    scatterer = td.Structure(
        geometry=transformed_geometry,
        medium=td.Medium(permittivity=eps),
    )

    # Simulation setup
    return td.Simulation(
        size=(2, 2, 2),
        run_time=1e-12,
        structures=[ scatterer, waveguide],
        sources=[mode_src],
        monitors=[mode_mnt],
        boundary_spec=td.BoundarySpec.all_sides(td.PML()),
        grid_spec=td.GridSpec.uniform(dl=dl),
    )


# Scaled Box

In [3]:
# def make_simulation(center: tuple, size: tuple, scale: tuple, eps: float) -> td.Simulation:
#     """Makes a simulation with a variable scatter width, height, and relative permittivity."""

#     wavelength = 1.0
#     freq0 = td.C_0 / wavelength
#     dl = 0.02

#     # Static structure
#     waveguide = td.Structure(
#         geometry=td.Box(size=(td.inf, 0.3, 0.2)), medium=td.Medium(permittivity=2.0)
#     )

#     # Forward source
#     mode_src = td.ModeSource(
#         size=(0, 1.5, 1.5),
#         center=(-0.9, 0, 0),
#         mode_index=0,
#         source_time=td.GaussianPulse(freq0=freq0, fwidth=freq0 / 10),
#         direction="+",
#     )

#     # Monitor
#     mode_mnt = td.ModeMonitor(
#         size=(0, 1.5, 1.5),
#         center=(+0.9, 0, 0),
#         mode_spec=mode_src.mode_spec,
#         freqs=[freq0],
#         name="mode",
#     )

#     # Geometry and transformation
#     scaled_size = (size[0] * scale[0], size[1] * scale[1], size[2] * scale[2])
#     transformed_geometry = td.Box(center=center, size=scaled_size)

#     # Scatterer structure
#     scatterer = td.Structure(
#         geometry=transformed_geometry,
#         medium=td.Medium(permittivity=eps),
#     )

#     # Simulation setup
#     return td.Simulation(
#         size=(2, 2, 2),
#         run_time=1e-12,
#         structures=[ scatterer, waveguide],
#         sources=[mode_src],
#         monitors=[mode_mnt],
#         boundary_spec=td.BoundarySpec.all_sides(td.PML()),
#         grid_spec=td.GridSpec.uniform(dl=dl),
#     )



Let's try setting up the simulation and plotting it for starters.

In [ ]:
# starting set of input parameters
center0 = (0.0, 0.0, 0.0)
size0 = (0.5, 1.0, 1.0)
scale0 = anp.array([1.2, 1.3, 1.4])
eps0 = 3.0

sim = make_simulation(center=center0, size=size0, scale = scale0, eps=eps0)
_, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, dim in zip(axes, "xyz"):
    sim.plot(**{dim: 0}, ax=ax)
plt.show()

### Post-processing the output data

After the simulation is run, it returns a `td.SimulationData` containing the data we want to post-process.

Let's write a function that will process this `td.SimulationData` and return the power in the mode amplitude of our output mode monitor.

In [5]:
def compute_power(sim_data: td.SimulationData) -> float:
    """Post process the result of the Simulation run to return the power in the mode at index=0."""

    freq0 = sim_data.simulation.monitors[0].freqs[0]
    mode_data = sim_data["mode"]
    mode_amps = mode_data.amps
    amp = mode_amps.sel(direction="+", f=freq0, mode_index=0).values
    return anp.sum(abs(amp) ** 2)

### Defining the tidy3d simulation function for differentiation

Next, we can import the `tidy3d..web.run` function and put all the pieces together into a single function to compute the 0th order transmitted power as a function of `center`, `size`, and `eps` (relative permittivty) of the scatterer.

In [6]:
def power(center, size, scale, eps):
    """Compute power transmitted into 0th order mode given scatterer parameters."""
    sim = make_simulation(center=center, size=size, scale=scale, eps=eps)
    sim_data = web.run(sim, task_name="autograd 1", local_gradient=True)
    return compute_power(sim_data)

In [ ]:
power(center0, size0, scale0, eps0)
# power(center0, size0, eps0)

### Running and differentiating the simulation using `autograd`

Finally, using the `autograd` tools described earlier, we can differentiate this `power` function. 

For demonstration, let's use `autograd.value_and_grad` to both compute the power **and** the gradient w.r.t. each of the 3 input parameters.

In [8]:
d_power = ag.value_and_grad(power, argnum=(0, 1, 2, 3))

We will run this function and assign variables to the power values and the gradients returned.

Note that running this will set off **two** separate tasks, one after another, called, `"adjoint_power_fwd"` and `"adjoint_power_adj"`.

The first is evaluating our simulation in "forward mode", computing the power and stashing information needed for gradient computation.

The second step runs the "adjoint" simulation, in which the output monitor is converted to a source and the simulation is re-run.

The results of both of these simulations runs are combined behind the scene to tell `autograd` how to compute the gradient for us.

In [ ]:
power_value, (dp_dcenter, dp_dsize, dp_dscale, dp_deps) = d_power(center0, size0, scale0, eps0)

> Note: the gradient evaluation functions returned by `autograd.grad()` do not accept keyword arguments (ie. `center=(0.,0.,0.)`) and instead accept positional arguments (without the argument name). You may run across this when trying to evaluate gradients so it's a good idea to keep in mind.

We can take a look at our computed power and gradient information. 

In [ ]:
print(f"power = {power_value:.3f}")
print(f"d_power/d_center = {dp_dcenter}")
print(f"d_power/d_size = {dp_dsize}")
print(f"d_power/d_scale = {dp_dscale}")
print(f"d_power/d_eps = {dp_deps}")

From this, we can infer several things that fit our intuition, for example that:
* the transmitted power should **decrease** if we increase the permittivity of our scatterer.
* the transmitted power does not depend strongly on the position of the scatterer along the propagation direction.

## Conclusion & Next Steps

This gives the most basic introduction to the principles behind the adjoint plugin.

In subsequent notebooks, we will show how to:
 * Check the gradients returned by this method against brute force computed gradients for accuracy.
 * Perform gradient-based optimization using the adjoint plugin.